# 🏠 Entrenamiento YOLOv8 para Planos CAD en Colab

Este notebook entrena un modelo YOLOv8 específico para detectar muebles en planos arquitectónicos CAD.

**Dataset:** Sintético CAD-like (generado aquí)  
**Tiempo estimado:** 20-30 minutos  
**GPU:** Tesla T4 (gratuito) ✅

## 1️⃣ Instalar dependencias

In [ ]:
!pip install -q ultralytics opencv-python pyyaml pillow numpy requests tqdm torch torchvision
print("✅ Dependencias instaladas")

## 2️⃣ Verificar GPU

In [ ]:
import torch
print(f"GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("⚠️ Sin GPU - el entrenamiento será más lento")

## 3️⃣ Montar Google Drive (para guardar modelo)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive montado")

## 4️⃣ Preparar Dataset en formato YOLO

In [ ]:
import yaml
import os
from pathlib import Path

# Crear estructura YOLO
dataset_path = Path('/content/dataset')
for split in ['train', 'val', 'test']:
    (dataset_path / split / 'images').mkdir(parents=True, exist_ok=True)
    (dataset_path / split / 'labels').mkdir(parents=True, exist_ok=True)

print("✅ Estructura de dataset creada")

# Clases para detectar (arquitectura + muebles)
classes = [
    'puerta',
    'ventana',
    'sofa',
    'cama',
    'mesa',
    'silla',
    'toilet',
    'sink',
    'bathtub',
    'shower',
    'armario',
    'escaleras',
    'electrodomestico',
    'puerta_ventana',
    'columna',
    'balcon'
]

# Crear data.yaml
data_yaml = {
    'path': '/content/dataset',
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'nc': len(classes),
    'names': classes
}

with open('/content/dataset/data.yaml', 'w') as f:
    yaml.dump(data_yaml, f)

print(f"✅ data.yaml creado con {len(classes)} clases")
print("Clases:", classes)

## 5️⃣ Generar Dataset Sintético CAD

In [ ]:
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm

dataset_path = Path('/content/dataset')

print("🎨 Generando dataset sintético CAD-like...")
np.random.seed(42)
n_images = 200  # Más imágenes = mejor entrenamiento
n_classes = len(classes)

for i in tqdm(range(n_images), desc="Generando imágenes"):
    # Crear imagen blanca (fondo CAD típico)
    img = np.ones((640, 640, 3), dtype=np.uint8) * 255
    
    # Dibujar muros (rectángulo exterior - líneas negras gruesas)
    cv2.rectangle(img, (50, 50), (590, 590), (0, 0, 0), 8)
    
    # Dibujar algunos muebles (rectángulos internos con líneas)
    n_objects = np.random.randint(2, 6)
    labels = []
    
    for _ in range(n_objects):
        x1 = np.random.randint(100, 500)
        y1 = np.random.randint(100, 500)
        w = np.random.randint(50, 150)
        h = np.random.randint(50, 150)
        x2 = min(x1 + w, 590)
        y2 = min(y1 + h, 590)
        
        # Dibujar rectángulo (mueble/objeto)
        cv2.rectangle(img, (x1, y1), (x2, y2), (100, 100, 100), 2)
        
        # Centro normalizado (formato YOLO: class_id cx cy w h)
        cx = ((x1 + x2) / 2) / 640
        cy = ((y1 + y2) / 2) / 640
        w_norm = (x2 - x1) / 640
        h_norm = (y2 - y1) / 640
        class_id = np.random.randint(0, n_classes)
        
        labels.append(f"{class_id} {cx:.6f} {cy:.6f} {w_norm:.6f} {h_norm:.6f}")
    
    # Dividir en train/val/test (70/15/15)
    if i < 140:
        split = 'train'
    elif i < 170:
        split = 'val'
    else:
        split = 'test'
    
    # Guardar imagen
    img_path = dataset_path / split / 'images' / f'image_{i:04d}.jpg'
    cv2.imwrite(str(img_path), img)
    
    # Guardar etiqueta
    label_path = dataset_path / split / 'labels' / f'image_{i:04d}.txt'
    with open(label_path, 'w') as f:
        f.write('\n'.join(labels))

print(f"\n✅ Dataset creado:")
print(f"   Train: 140 imágenes")
print(f"   Val: 30 imágenes")
print(f"   Test: 30 imágenes")
print(f"   Total: {n_images} imágenes (640x640)")

## 6️⃣ Entrenar YOLOv8

In [ ]:
from ultralytics import YOLO
import torch

print("🤖 Cargando YOLOv8 Medium...")
model = YOLO('yolov8m.pt')

print("\n🚀 Iniciando entrenamiento...")
print("="*50)
print("Parámetros:")
print("  - Modelo: YOLOv8 Medium")
print("  - Épocas: 50")
print("  - Batch size: 16")
print("  - Imagen: 640x640")
print("  - Clases: 16")
print("  - Dataset: 200 imágenes sintéticas CAD")
print("="*50)
print()

# Entrenar
results = model.train(
    data='/content/dataset/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,  # Early stopping
    device=0 if torch.cuda.is_available() else 'cpu',
    project='/content/runs',
    name='floorplan_local',
    exist_ok=True,
    verbose=True,
    save=True
)

print("\n✅ Entrenamiento completado!")

## 7️⃣ Validar modelo

In [ ]:
print("🧪 Validando modelo...")
metrics = model.val()

print(f"\n📊 Resultados de validación:")
print(f"  - mAP50: {metrics.box.map50:.4f}")
print(f"  - mAP50-95: {metrics.box.map:.4f}")
print(f"  - Precision: {metrics.box.mp:.4f}")
print(f"  - Recall: {metrics.box.mr:.4f}")

## 8️⃣ Guardar modelo en Google Drive

In [ ]:
import shutil
from pathlib import Path

# Ruta del modelo entrenado
model_source = Path('/content/runs/floorplan_local/weights/best.pt')
drive_path = Path('/content/drive/MyDrive/best_floorplancad_colab.pt')

if model_source.exists():
    # Copiar a Google Drive
    shutil.copy(model_source, drive_path)
    size_mb = drive_path.stat().st_size / 1e6
    print(f"✅ Modelo guardado en Google Drive")
    print(f"   Ruta: {drive_path}")
    print(f"   Tamaño: {size_mb:.1f} MB")
else:
    print(f"❌ Modelo no encontrado: {model_source}")

## 9️⃣ Descargar directamente desde Colab

In [ ]:
from google.colab import files
from pathlib import Path

model_path = '/content/runs/floorplan_local/weights/best.pt'

if Path(model_path).exists():
    size_mb = Path(model_path).stat().st_size / 1e6
    print(f"📥 Descargando modelo ({size_mb:.1f} MB)...")
    files.download(model_path)
    print("✅ Descarga completada!")
else:
    print("❌ Modelo no encontrado")

## 🔟 Probar en imágenes de test

In [ ]:
print("🔍 Probando modelo en imágenes de test...")

results = model.predict(
    source='/content/dataset/test/images',
    conf=0.35,
    save=True,
    project='/content/runs'
)

print(f"\n✅ Predicción completada!")
print(f"Resultados guardados en: /content/runs/detect")

## ✨ Instrucciones finales

1. **El modelo fue entrenado y guardado:**
   - En Google Drive: `best_floorplancad_colab.pt`
   - Descarga directa desde Colab: `best.pt`

2. **Descargalo y cópialo a tu Backend-Python:**
   ```bash
   cp ~/Downloads/best.pt /home/alan/Projects/Software2/Parcial1/Backend-Python/YOLO/best_floorplancad_colab.pt
   ```

3. **Actualiza `app/core/config.py`:**
   ```python
   YOLO_MODEL_PATH = "YOLO/best_floorplancad_colab.pt"
   ```

4. **Reinicia el servidor:**
   ```bash
   cd /home/alan/Projects/Software2/Parcial1/Backend-Python
   source .venv/bin/activate
   uvicorn app.main:app --reload
   ```

5. **Prueba con tus imágenes CAD:**
   ```bash
   python3 test_api.py --image "imagenes/WhatsApp Image 2026-09-21 at 5.24.05 PM.jpeg"
   ```